In [1]:
# importando bibliotecas
from faker import Faker
from random import choice, choices, randint, triangular, uniform
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from unicodedata import normalize
from re import sub, IGNORECASE

In [2]:
# Carregando variáveis de ambiente
load_dotenv()

# Acessando o banco de dados
DB_URL = os.getenv("DB_URL")
engine = create_engine(DB_URL)
faker = Faker('pt_BR')

In [3]:
def apenas_numeros(valor: str) -> str:
    return sub(r'\D', '', valor)

def remove_acentos(texto: str) -> str:
    return normalize('NFKD', texto.lower()).encode('ASCII', 'ignore').decode('ASCII')

In [4]:
# Constantes
QTD_LOJAS               = 10
QTD_BANCOS              = 5
QTD_CONTAS              = 80
QTD_PAGAMENTOS          = 1000
QTD_VENDAS              = 1000
QTD_TRANSPORTADORAS     = 8
QTD_FUNCIONARIOS        = 50
QTD_ENVIA_ITENS         = 3000
QTD_CONTA_LOJA          = 15
QTD_CONTA_FUNCIONARIO   = 60
QTD_PAGAMENTO_PARCELADO = 300

In [5]:
dados_loja = []
for i in range(QTD_LOJAS):
    cnpj_formatado = apenas_numeros(faker.cnpj())
    telefone_formatado = apenas_numeros(faker.cellphone_number())
    
    dados_loja.append({
        'nome': faker.company(),
        'cnpj': cnpj_formatado,
        'email': faker.email(),
        'telefone': telefone_formatado,
        'percentual_comissao': round(uniform(0.01, 0.2), 2)
    })

df_lojas = pd.DataFrame(dados_loja)
display(df_lojas.head())

,nome,cnpj,email,telefone,percentual_comissao
0,Lima,63527914000125,bryanda-cunha@example.com,5504999590695,0.16
1,Câmara,49735608000153,maria-fernandanovais@example.com,5586980614310,0.13
2,Fernandes Ltda.,94063875000186,pedro-lucasnovaes@example.com,5517948684900,0.16
3,Lopes,23907615000100,benjamimnovais@example.com,5563951966620,0.17
4,Cassiano e Filhos,01846392000177,hfernandes@example.com,55082969727524,0.14


In [6]:
dados_banco = []
for i in range(QTD_BANCOS):
    cnpj_formatado = apenas_numeros(faker.cnpj())
    
    dados_banco.append({
        'nome': f'Banco {faker.company()}',
        'cnpj': cnpj_formatado
    })

df_bancos = pd.DataFrame(dados_banco)
display(df_bancos.head())

,nome,cnpj
0,Banco Ferreira,79831256000109
1,Banco Lopes Guerra S.A.,82931450000162
2,Banco Andrade - ME,23517609000138
3,Banco Porto Ltda.,12790438000108
4,Banco Ribeiro,45701296000106


In [7]:
dados_conta = []

TIPOS_CONTA = ['CORRENTE', 'POUPANCA', 'SALARIO']

for i in range(QTD_CONTAS):
    dados_conta.append({
        'agencia': str(randint(1, 9999)).zfill(4),
        'numero_conta': str(randint(1, 999999)).zfill(6),
        'tipo_conta': choice(TIPOS_CONTA),
        'saldo': round(triangular(0, 9999.99, 100), 2),
        'cod_banco': randint(1, QTD_BANCOS)
    })

df_contas = pd.DataFrame(dados_conta)
display(df_contas.head())

,agencia,numero_conta,tipo_conta,saldo,cod_banco
0,1716,075168,SALARIO,945.01,2
1,4532,863409,SALARIO,2471.63,5
2,9398,583001,CORRENTE,7730.52,2
3,5324,394652,POUPANCA,3145.07,1
4,9947,699630,CORRENTE,913.84,5


In [8]:
dados_pagamento = []

TIPOS_PAGAMENTO = ['BOLETO', 'PIX', 'DEBITO', 'CREDITO']
STATUS_PAGAMENTO = ['AGUARDANDO', 'CONCLUIDO', 'EXPIRADO']

for i in range(QTD_PAGAMENTOS):
    cod_conta_origem = randint(1, QTD_CONTAS)
    cod_conta_destino = randint(1, QTD_CONTAS)
    while cod_conta_destino == cod_conta_origem:
        cod_conta_destino = randint(1, QTD_CONTAS)
    
    dados_pagamento.append({
        'tipo_pagamento': choice(TIPOS_PAGAMENTO),
        'status_pagamento': choice(STATUS_PAGAMENTO),
        'valor': round(triangular(0, 999.99, 100), 2),
        'cod_conta_origem': cod_conta_origem,
        'cod_conta_destino': cod_conta_destino
    })

df_pagamentos = pd.DataFrame(dados_pagamento)
display(df_pagamentos.head())

,tipo_pagamento,status_pagamento,valor,cod_conta_origem,cod_conta_destino
0,PIX,AGUARDANDO,241.01,18,44
1,CREDITO,AGUARDANDO,168.26,10,73
2,BOLETO,AGUARDANDO,682.87,55,56
3,PIX,CONCLUIDO,774.44,37,13
4,DEBITO,CONCLUIDO,651.45,19,77


In [ ]:
dados_funcionario = []
for i in range(QTD_FUNCIONARIOS):
    nome = sub(r'\b(sr|sra|srta|dr|dra)\.\s*', '', faker.name(), flags=IGNORECASE)
    cpf_formatado = apenas_numeros(faker.cpf())
    telefone_formatado = apenas_numeros(faker.cellphone_number())
    email = f'{sub(r' ', '_', remove_acentos(nome))}@gmail.com'
    
    dados_funcionario.append({
        'nome': nome,
        'cpf': cpf_formatado,
        'cargo': 'Vendedor',
        'salario': round(triangular(1621, 4000, 100), 2),
        'email': email,
        'telefone': telefone_formatado,
        'cod_loja': randint(1, QTD_LOJAS)
    })

df_funcionarios = pd.DataFrame(dados_funcionario)
display(df_funcionarios.head())

,nome,cpf,cargo,salario,email,telefone,cod_loja
0,Ágatha Correia,19764253016,Profissional de relações públicas,3771.50,agatha_correia@gmail.com,5523913316813,5
1,Lunna Macedo,57093412661,Engenheiro de materiais,1610.03,lunna_macedo@gmail.com,5503923950382,8
2,Calebe Ramos,82094735141,Mestre-de-obras,1146.66,calebe_ramos@gmail.com,5557943269329,9
3,Eloah Costa,57426389065,Selecionador de pessoal,1886.36,eloah_costa@gmail.com,5563965777229,7
4,André Montenegro,38741260996,Cortador de cana-de-açucar,1907.21,andre_montenegro@gmail.com,5594954677529,8


In [10]:
dados_venda = []

STATUS_VENDA = ['APROVADO', 'CANCELADO', 'ESTORNADO']

for i in range(QTD_VENDAS):
    cod_vendedor = randint(1, QTD_FUNCIONARIOS)
    cod_loja = df_funcionarios.loc[
        cod_vendedor - 1, 'cod_loja'
    ]
    
    dados_venda.append({
        'valor_total': round(triangular(0, 999.99, 100), 2),
        'status_venda': choices(STATUS_VENDA, weights=[85, 5, 10], k=1)[0],
        'cod_vendedor': cod_vendedor,
        'cod_loja': cod_loja
    })

df_vendas = pd.DataFrame(dados_venda)
display(df_vendas.head())

,valor_total,status_venda,cod_vendedor,cod_loja
0,92.95,ESTORNADO,25,3
1,98.79,APROVADO,44,3
2,615.36,APROVADO,11,1
3,36.84,APROVADO,24,5
4,386.27,APROVADO,46,4


In [11]:
dados_transportadora = []
for i in range(QTD_TRANSPORTADORAS):
    cnpj_formatado = apenas_numeros(faker.cnpj())
    telefone_formatado = apenas_numeros(faker.cellphone_number())
    
    dados_transportadora.append({
        'nome': faker.company(),
        'cnpj': cnpj_formatado,
        'telefone': telefone_formatado,
        'email': faker.email(),
        'taxa_entrega': round(uniform(0.01, 1), 2)
    })

df_transportadoras = pd.DataFrame(dados_transportadora)
display(df_transportadoras.head())

,nome,cnpj,telefone,email,taxa_entrega
0,Rios Souza - EI,20943186000120,5546997410668,cauacostela@example.org,0.71
1,Melo,95671042000160,55096959168264,melinacassiano@example.org,0.16
2,Pacheco - EI,14520389000100,5501920900428,tmoraes@example.com,0.74
3,Fonseca,30524678000134,5555908299560,stellajesus@example.com,0.65
4,Novaes Rezende S/A,40586712000159,5578942341981,theodoroda-mata@example.org,0.71


In [12]:
dados_envia_itens = []
dados_conta_loja = []
dados_conta_funcionario = []
dados_pagamento_parcelado = []

for i in range(QTD_ENVIA_ITENS):
    dados_envia_itens.append({
        'cod_loja': randint(1, QTD_LOJAS),
        'cod_transportadora': randint(1, QTD_TRANSPORTADORAS)
    })

for i in range(QTD_CONTA_LOJA):
    dados_conta_loja.append({
        'cod_loja': randint(1, QTD_LOJAS),
        'cod_conta': randint(1, QTD_CONTAS)
    })

for i in range(QTD_CONTA_FUNCIONARIO):
    dados_conta_funcionario.append({
        'cod_funcionario': randint(1, QTD_FUNCIONARIOS),
        'cod_conta': randint(1, QTD_CONTAS)
    })

for i in range(QTD_PAGAMENTO_PARCELADO):
    dados_pagamento_parcelado.append({
        'cod_pagamento': randint(1, QTD_PAGAMENTOS),
        'cod_venda': randint(1, QTD_VENDAS)
    })

df_envia_itens = pd.DataFrame(dados_envia_itens)
df_conta_loja = pd.DataFrame(dados_conta_loja)
df_conta_funcionario = pd.DataFrame(dados_conta_funcionario)
df_pagamento_parcelado = pd.DataFrame(dados_pagamento_parcelado)

In [13]:
# Inserindo no banco de dados
tabelas = {
    'tb_loja': df_lojas,
    'tb_banco': df_bancos,
    'tb_transportadora': df_transportadoras,
    'tb_funcionario': df_funcionarios,
    'tb_conta': df_contas,
    'tb_pagamento': df_pagamentos,
    'tb_venda': df_vendas,
    'tb_envia_itens': df_envia_itens,
    'tb_conta_loja': df_conta_loja,
    'tb_conta_funcionario': df_conta_funcionario,
    'tb_pagamento_parcelado': df_pagamento_parcelado
}

print('Iniciando conexão com o banco de dados')
for t, df in tabelas.items():
    try:
        df.to_sql(name=t, con=engine, if_exists='append', index=False)
    except Exception as e:
        print(f'Um erro no banco aconteceu')
        print(e)
print('Conexão feita com sucesso')

Iniciando conexão com o banco de dados
Conexão feita com sucesso
